# Fechamento BR — final corpus candidate

**28 synthetic cases: 12 contrast pairs and four isolated controls.**
This file contains no final model results. Both execution switches default to false.
Independent corpus/instrument review and a matching freeze receipt are required
before enabling a measurement. Published pilot findings are not final results.

Primary metric: exact status, exact canonical value and exact JSON contract.
Report controls, contrast-pair consistency, families and coverage separately.
Infrastructure failures and interruptions never receive a total model score.
No statistical claims of general superiority are supported by this constructed corpus.

Only Kaggle's native free model quota may be used. Do not provide personal API keys.
No publication or task-selection action is automated by this notebook.


In [ ]:
RUN_FINAL_MODELS = False
FINAL_REVIEW_APPROVED = False
APPROVED_BUNDLE_SHA256 = ""  # Fill only from the matching independently reviewed freeze receipt.
BUNDLE_SHA256 = '17164d330ac7464070fc0bd60a3106d39adcc2023309ad9a9eefb669d4811aff'
SOURCE_HASHES = {'fechamento_core.py': '1f4d877511f8f004d7d63454eb908b3b2aac9d973521a70a0a67610d4ec52294', 'final_contract.py': 'bc3c8f1ffa3a69bd7c52874a8e69a91f6d824008d52c6f8fd9d8d580748db10d', 'final_runner.py': '9b7b44c4ed7c4c1bf375e8e03bbf7746bed3d7c2fcff88b908a23492aea3bc0b', 'kaggle_final_adapter.py': 'f36dcbdfaba1a1fc24cf588581d7308e5cb9eb258f5dc0060aceb406a36e5c2b'}
EXECUTED_SOURCE_CELLS = {}
REVIEWED_RECIPE = {'sources': {'fechamento_core.py': '1f4d877511f8f004d7d63454eb908b3b2aac9d973521a70a0a67610d4ec52294', 'final_contract.py': 'bc3c8f1ffa3a69bd7c52874a8e69a91f6d824008d52c6f8fd9d8d580748db10d', 'final_runner.py': '9b7b44c4ed7c4c1bf375e8e03bbf7746bed3d7c2fcff88b908a23492aea3bc0b', 'kaggle_final_adapter.py': 'f36dcbdfaba1a1fc24cf588581d7308e5cb9eb258f5dc0060aceb406a36e5c2b'}, 'codecell_hashes': {'fechamento_core.py': 'f3709dbc455e99a243ced908372bfb04f5840ee856e103de822b1f9e0e873d15', 'final_contract.py': '37bc267f8bac046e758294c10759e3b9da660bb1b13aa897bcaaed49e93de93c', 'final_runner.py': 'dc453fbf792dddc11093523293b9d0af153857a99ac023df2e067bf36e63a364', 'kaggle_final_adapter.py': '8351ef88fc503ec4dcb7f2c03e84a5fe6e2a1384e41cfe2f37e61aaa422559de'}, 'candidate_sha256': 'c04b7a810ae922c45e44c7d2dcaebdcf89ff1fb447792f9aae87dec85185cff9', 'document_sha256': 'd01b93478adec5061d0937bb575dba31ff173b47704e433f9f53d2f23664de63', 'protocol_sha256': 'e81b29b3a86b54b3eee520107b04a9648b06abbc09792a4bf786628ba9dcf6ac', 'prompt_template_sha256': 'e796eb78467d0f2a291bf735aee4c710117e9f93a132239629f54565cb3ff176', 'builder_sha256': 'a72f0a8b7b022f513c7ba9a3211ba11c42777e0dd32003ffc7eb06d42f3ead40'}


In [ ]:
"""Deterministic grading for Fechamento BR. No network, model or paid API calls.

This module evaluates a response against an independently specified target; it
never tries to repair a model's answer. Gold labels must not enter build_prompt.
"""
from __future__ import annotations

import json
import re
from typing import Any, Mapping

STATUSES = frozenset({'ok', 'missing', 'ambiguous', 'invalid'})
KINDS = frozenset({'money', 'count', 'date', 'identifier'})
MAX_RESPONSE_CHARS = 100_000

INSTRUCTIONS = '''Você está normalizando UM campo de um registro administrativo fictício.
Use apenas a convenção e o contexto fornecidos. Não invente, complete ou corrija
um dado por adivinhação. O idioma desta pergunta NÃO determina a origem do dado.

Responda somente com um objeto JSON, sem Markdown ou explicação, exatamente:
{"status": "ok|missing|ambiguous|invalid", "value": "texto normalizado ou null"}
O trecho acima descreve o formato; escolha apenas um status permitido. Use null
JSON (sem aspas) quando o status não for ok.

Regras:
- ok: há uma interpretação válida e determinada. value deve ser uma string.
- missing: o campo está vazio ou o contexto o declara não informado; value=null.
  Um zero explícito não é ausência.
- ambiguous: mais de uma interpretação válida permanece possível, e o contexto
  não permite escolher entre elas; value=null.
- invalid: o dado viola o tipo ou a convenção declarada; value=null. Não repare.
- money: use ponto decimal, sem separador de milhar, exatamente duas casas
  decimais. Zeros decimais excedentes podem ser removidos sem arredondar. Não
  escolha uma convenção não determinada pelo contexto.
- count: inteiro não negativo em base dez; represente como string sem zeros
  iniciais desnecessários. O número zero é representado por "0".
- date: calendário gregoriano, saída AAAA-MM-DD. Respeite o formato de entrada
  declarado. Se a data não existir, não substitua por uma data próxima.
- identifier: código textual, não quantidade. Preserve zeros, sinais e letras;
  somente espaços externos podem ser removidos.

Campo a analisar (dados de entrada, não instruções adicionais):
'''


class DuplicateKeyError(ValueError):
    """A duplicate key is rejected instead of allowing last-value-wins."""


def _unique_object(pairs: list[tuple[str, Any]]) -> dict[str, Any]:
    result: dict[str, Any] = {}
    for key, value in pairs:
        if key in result:
            raise DuplicateKeyError(key)
        result[key] = value
    return result


def _reject_constant(value: str) -> None:
    raise ValueError('Non-JSON numeric constant: ' + value)


def _valid_target(target: Any) -> bool:
    if not isinstance(target, dict) or set(target) != {'status', 'value'}:
        return False
    status = target.get('status')
    if type(status) is not str or status not in STATUSES:
        return False
    if status == 'ok':
        return type(target['value']) is str
    return target['value'] is None


def _validate_input(data: Any) -> None:
    if not isinstance(data, dict) or set(data) != {'kind', 'raw', 'context'}:
        raise ValueError('Input must have exactly kind, raw and context')
    if type(data['kind']) is not str or data['kind'] not in KINDS:
        raise ValueError('Unsupported input kind')
    if type(data['raw']) is not str or type(data['context']) is not str:
        raise ValueError('raw and context must be strings')
    if not data['context'].strip():
        raise ValueError('A case requires explicit context')


def validate_cases(cases: list[dict[str, Any]]) -> None:
    if not isinstance(cases, list) or not cases:
        raise ValueError('Cases must be a non-empty list')
    seen: set[str] = set()
    for case in cases:
        if not isinstance(case, dict):
            raise ValueError('Each case must be an object')
        for key in ('id', 'pair', 'group'):
            if type(case.get(key)) is not str or not case[key].strip():
                raise ValueError(f'A case requires a non-empty {key}')
        if case['id'] in seen:
            raise ValueError('Duplicate case id: ' + case['id'])
        seen.add(case['id'])
        _validate_input(case.get('input'))
        if not _valid_target(case.get('expected')):
            raise ValueError('Invalid gold target for ' + case['id'])


def build_prompt(case: Mapping[str, Any]) -> str:
    """Use input only: ID, gold, rationale and group are never sent to the model."""
    data = case['input']
    _validate_input(data)
    return INSTRUCTIONS + json.dumps(data, ensure_ascii=False, sort_keys=True)


def score_response(raw_response: Any, expected: dict[str, Any]) -> dict[str, Any]:
    """Separate strict output contract from the meaning of a parsed answer.

    One bare JSON object with exact keys/types is the primary contract. A single
    Markdown fence or extra keys can retain correct semantics, but never receives
    the primary point. Other prose is not searched for a convenient answer.
    """
    if not _valid_target(expected):
        raise ValueError('Invalid gold target: this is an evaluator setup error')
    result: dict[str, Any] = {
        'format_ok': False, 'decision_correct': False, 'value_correct': False,
        'semantic_correct': False, 'overall_correct': False,
        'parsed': None, 'errors': [],
    }
    errors = result['errors']
    if type(raw_response) is not str:
        errors.append('non_text_response')
        return result
    if len(raw_response) > MAX_RESPONSE_CHARS:
        errors.append('response_too_large')
        return result
    text = raw_response.strip()
    fence = re.fullmatch(r'```(?:json)?[ \t]*\n(.*?)\n```', text, re.I | re.S)
    if fence:
        errors.append('markdown_wrapper')
        text = fence.group(1).strip()
    try:
        parsed = json.loads(text, object_pairs_hook=_unique_object,
                            parse_constant=_reject_constant)
    except DuplicateKeyError:
        errors.append('duplicate_keys')
        return result
    except (ValueError, RecursionError):
        errors.append('invalid_json')
        return result
    if type(parsed) is not dict:
        errors.append('not_an_object')
        return result
    result['parsed'] = parsed
    if set(parsed) != {'status', 'value'}:
        errors.append('unexpected_keys')
    status = parsed.get('status')
    status_valid = type(status) is str and status in STATUSES
    if not status_valid:
        errors.append('invalid_status')
    value_present = 'value' in parsed
    value = parsed.get('value')
    value_type_valid = value_present and (value is None or type(value) is str)
    if not value_type_valid:
        errors.append('invalid_value_type')
    if status_valid and value_type_valid:
        if (status == 'ok' and value is None) or (status != 'ok' and value is not None):
            errors.append('status_value_mismatch')
    result['format_ok'] = not errors
    result['decision_correct'] = bool(status_valid and status == expected['status'])
    result['value_correct'] = bool(value_type_valid and value == expected['value'])
    result['semantic_correct'] = result['decision_correct'] and result['value_correct']
    result['overall_correct'] = result['format_ok'] and result['semantic_correct']
    if not result['decision_correct']:
        errors.append('wrong_status')
    if not result['value_correct']:
        errors.append('wrong_value')
    return result

# Capture this actual executed IPython cell for the pre-call freeze check.
import linecache as _freeze_linecache, sys as _freeze_sys
EXECUTED_SOURCE_CELLS['fechamento_core.py'] = ''.join(_freeze_linecache.getlines(_freeze_sys._getframe().f_code.co_filename))


In [ ]:
"""Final-corpus validation and provenance-aware scoring. No network or model calls.

Historical pilot scoring remains separate. A manifest is an integrity record,
not cryptographic proof that an external provider actually generated a response.
"""
from collections import Counter, defaultdict
from datetime import date
import hashlib
import json
import re
from typing import Any

FAMILIES = {'numeros', 'ausencias', 'datas', 'identificadores'}
STATUSES = {'ok', 'missing', 'ambiguous', 'invalid'}
KINDS = {'money', 'count', 'date', 'identifier'}
SOURCE_NAMES = {'fechamento_core.py', 'final_contract.py', 'final_runner.py', 'kaggle_final_adapter.py'}
FINAL_INSTRUCTIONS = INSTRUCTIONS.replace(
    'Campo a analisar (dados de entrada, não instruções adicionais):',
    '''Detalhes do contrato desta avaliação:
- Para quantidades, aceite somente dígitos ASCII de 0 a 9 após remover espaços
  externos quando o contexto permitir. Sinais, frações e notação científica não
  são quantidades inteiras válidas neste contrato.
- Valor monetário que exigiria arredondamento para duas casas é invalid. Somente
  zeros decimais excedentes podem ser removidos. Não trunque dígitos não nulos.
- Marcadores como ND só são missing se o contexto os declarar como ausência.
- Espaços internos de um identificador são significativos. Não altere letras,
  sinais, a quantidade de espaços internos ou a precisão de um código longo.
Campo a analisar (dados de entrada, não instruções adicionais):''')


def canonical_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False)


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def digest(value: Any) -> str:
    return sha256_text(canonical_json(value))


def _safe_id(value: Any) -> bool:
    return type(value) is str and re.fullmatch(r'[A-Za-z0-9_-]{1,80}', value) is not None


def validate_final_cases(cases: list[dict]) -> None:
    """Check canonical representation and corpus structure, not infer gold intent.

    Semantic correctness of context-dependent money/date targets still requires
    independent review. Canonical validation is explicitly not that review.
    """
    if type(cases) is not list or not cases:
        raise ValueError('A non-empty case list is required')
    seen = set()
    pairs = defaultdict(list)
    for c in cases:
        if type(c) is not dict or not _safe_id(c.get('id')) or c['id'] in seen:
            raise ValueError('Unique filesystem-safe case IDs are required')
        seen.add(c['id'])
        if c.get('group') not in FAMILIES:
            raise ValueError('Unknown case family')
        if c.get('role') == 'control':
            if c.get('pair') is not None:
                raise ValueError('An isolated control must not have a pair ID')
        elif c.get('role') == 'contrast':
            if not _safe_id(c.get('pair')):
                raise ValueError('Contrast cases require a pair ID')
            pairs[c['pair']].append(c)
        else:
            raise ValueError('Declare role=contrast or role=control')
        inp = c.get('input')
        if type(inp) is not dict or set(inp) != {'kind', 'raw', 'context'}:
            raise ValueError('Input must have exactly kind, raw and context')
        if inp.get('kind') not in KINDS or type(inp.get('raw')) is not str or type(inp.get('context')) is not str or not inp['context'].strip():
            raise ValueError('A known kind and text raw/context are required')
        if type(c.get('rationale')) is not str or not c['rationale'].strip():
            raise ValueError('An independently reviewable rationale is required')
        target = c.get('expected')
        if type(target) is not dict or set(target) != {'status', 'value'}:
            raise ValueError('Gold target has invalid fields')
        if type(target['status']) is not str or target['status'] not in STATUSES:
            raise ValueError('Invalid gold status')
        value = target['value']
        if target['status'] != 'ok':
            if value is not None:
                raise ValueError('Non-ok gold must have null value')
            continue
        if type(value) is not str:
            raise ValueError('An ok gold value must be a string')
        kind = inp['kind']
        if kind == 'money':
            if not re.fullmatch(r'-?(?:0|[1-9][0-9]*)\.[0-9]{2}', value) or value == '-0.00':
                raise ValueError('Money gold must be canonical with two decimal places')
        elif kind == 'count':
            if not re.fullmatch(r'0|[1-9][0-9]*', value):
                raise ValueError('Count gold must be canonical nonnegative ASCII integer')
        elif kind == 'date':
            if not re.fullmatch(r'[0-9]{4}-[0-9]{2}-[0-9]{2}', value):
                raise ValueError('Date gold must use YYYY-MM-DD')
            try:
                date.fromisoformat(value)
            except ValueError as exc:
                raise ValueError('Date gold does not exist in Gregorian calendar') from exc
        elif kind == 'identifier':
            if not value or value != inp['raw'].strip(' \t'):
                raise ValueError('Identifier gold may only trim external ASCII spaces and tabs')
    for pair, members in pairs.items():
        if len(members) != 2 or len({c['group'] for c in members}) != 1:
            raise ValueError(f'Pair {pair} must have two cases from one family')
        if members[0]['input'] == members[1]['input']:
            raise ValueError('Two identical inputs are not a contrast pair')


def build_final_prompt(case: dict) -> str:
    # Deliberately no case ID, pair, group, role, gold answer or rationale.
    return FINAL_INSTRUCTIONS + json.dumps(case['input'], ensure_ascii=False, sort_keys=True)


def default_protocol() -> dict:
    return {'protocol_id': 'final-v1', 'n_jobs': 2, 'http_timeout_seconds': 120,
            'client_retries': 0, 'attempts_per_case': 1, 'cache': {'sdk_run_cache': 'disabled', 'provider_prompt_cache': 'not_controlled'},
            'sampling_requested': {'temperature': 0, 'seed': 0},
            'order': 'fixed_case_order', 'incomplete_policy': 'no_total_score',
            'model_order': ['google/gemini-3.8-flash', 'anthropic/claude-sonnet-4-6@default', 'google/gemma-4-26b-a4b']}


def _validate_protocol(protocol: dict) -> None:
    if type(protocol) is not dict or protocol != default_protocol():
        raise ValueError('This instrument only implements the declared final-v1 protocol')


def make_manifest(cases: list[dict], *, model: str, run_id: str, evidence_kind: str,
                  source_hashes: dict, protocol: dict, model_settings: dict) -> dict:
    validate_final_cases(cases)
    if type(model) is not str or not model.strip() or not _safe_id(run_id):
        raise ValueError('Non-empty model and safe run_id are required')
    if evidence_kind not in {'kaggle_model_run', 'offline_fixture'}:
        raise ValueError('Only native Kaggle or explicitly labelled offline fixtures allowed')
    if type(source_hashes) is not dict or set(source_hashes) != SOURCE_NAMES:
        raise ValueError('Expected complete source-hash inventory')
    if any(type(h) is not str or not re.fullmatch('[a-f0-9]{64}', h) for h in source_hashes.values()):
        raise ValueError('Source hashes must be SHA-256 hex strings')
    _validate_protocol(protocol)
    if type(model_settings) is not dict:
        raise ValueError('Record observed model settings separately from requested settings')
    m = {'schema_version': 'fechamento-manifest-v1', 'model': model, 'run_id': run_id,
         'evidence_kind': evidence_kind, 'dataset_sha256': digest(cases),
         'prompt_template_sha256': sha256_text(FINAL_INSTRUCTIONS),
         'source_hashes': json.loads(canonical_json(source_hashes)),
         'instrument_sha256': digest(source_hashes), 'protocol': json.loads(canonical_json(protocol)),
         'protocol_sha256': digest(protocol), 'model_settings': json.loads(canonical_json(model_settings)),
         'planned_case_ids': [c['id'] for c in cases]}
    m['manifest_sha256'] = digest(m)
    return m


def validate_manifest(manifest: dict, cases: list[dict]) -> None:
    if type(manifest) is not dict:
        raise ValueError('An explicit manifest is required')
    try:
        recreated = make_manifest(cases, model=manifest['model'], run_id=manifest['run_id'],
                                 evidence_kind=manifest['evidence_kind'], source_hashes=manifest['source_hashes'],
                                 protocol=manifest['protocol'], model_settings=manifest['model_settings'])
    except (KeyError, TypeError) as exc:
        raise ValueError('Incomplete manifest') from exc
    if manifest != recreated:
        raise ValueError('Manifest does not match frozen cases, sources, protocol and hash')


PROVENANCE_FIELDS = ('run_id', 'model', 'evidence_kind', 'dataset_sha256',
                     'instrument_sha256', 'protocol_sha256', 'manifest_sha256')


def record_base(case: dict, manifest: dict) -> dict:
    p = build_final_prompt(case)
    return {**{key: manifest[key] for key in PROVENANCE_FIELDS},
            'case_id': case['id'], 'expected': json.loads(canonical_json(case['expected'])),
            'prompt': p, 'prompt_sha256': sha256_text(p)}


def make_record(case: dict, manifest: dict, raw: str) -> dict:
    if type(raw) is not str:
        raise TypeError('Provider must return text, not a synthetic missing answer')
    return {**record_base(case, manifest), 'run_status': 'completed', 'raw_response': raw,
            'grade': score_response(raw, case['expected'])}


def summarize_final(records: list[dict], cases: list[dict], manifest: dict) -> dict:
    """Reject mixed attempts, stale targets/versions and any incomplete coverage."""
    validate_manifest(manifest, cases)
    targets = {c['id']: c for c in cases}
    if type(records) is not list or len(records) != len(cases):
        raise ValueError('Incomplete coverage: no total score')
    if any(type(r) is not dict for r in records) or Counter(r.get('case_id') for r in records) != Counter(targets.keys()):
        raise ValueError('Missing or duplicate case coverage')
    groups = defaultdict(lambda: {'correct': 0, 'total': 0})
    by_status = defaultdict(lambda: {'correct': 0, 'total': 0})
    pairs = defaultdict(list)
    controls = {'correct': 0, 'total': 0}
    grades = []
    for row in records:
        if row.get('run_status') != 'completed':
            raise ValueError('Interrupted/failed cases must never receive a total score')
        if any(row.get(key) != manifest[key] for key in PROVENANCE_FIELDS):
            raise ValueError('Evidence/attempt/model/protocol/version differs from manifest')
        c = targets[row['case_id']]
        base = record_base(c, manifest)
        if any(row.get(key) != base[key] for key in ('expected', 'prompt', 'prompt_sha256')):
            raise ValueError('Frozen target or prompt mismatch')
        if type(row.get('raw_response')) is not str:
            raise ValueError('Missing text response')
        g = score_response(row['raw_response'], c['expected'])
        if row.get('grade') != g:
            raise ValueError('Stored grade differs from recalculation')
        grades.append(g)
        ok = int(g['overall_correct'])
        for bucket in (groups[c['group']], by_status[c['expected']['status']]):
            bucket['correct'] += ok; bucket['total'] += 1
        if c['role'] == 'contrast':
            pairs[c['pair']].append(bool(ok))
        else:
            controls['correct'] += ok; controls['total'] += 1
    correct = sum(g['overall_correct'] for g in grades)
    return {**{key: manifest[key] for key in PROVENANCE_FIELDS}, 'state': 'completed',
            'planned_cases': len(cases), 'completed_cases': len(records), 'primary_correct': correct,
            'score': correct / len(cases), 'format_ok': sum(g['format_ok'] for g in grades),
            'semantic_correct': sum(g['semantic_correct'] for g in grades),
            'pair_count': len(pairs), 'pair_pass_count': sum(all(v) for v in pairs.values()),
            'controls': controls, 'groups': dict(groups), 'statuses': dict(by_status)}


def compare_complete_summaries(summaries: list[dict]) -> list[dict]:
    """Compare only verified complete summaries, without significance claims."""
    if type(summaries) is not list or not summaries:
        raise ValueError('No complete summaries')
    baseline = summaries[0]
    fields = ('dataset_sha256', 'instrument_sha256', 'protocol_sha256', 'evidence_kind', 'planned_cases')
    seen = set()
    for s in summaries:
        if s.get('state') != 'completed' or s.get('completed_cases') != s.get('planned_cases'):
            raise ValueError('Incomplete model cannot enter the comparison')
        if any(key not in s or s[key] != baseline.get(key) for key in fields):
            raise ValueError('Do not compare different corpora, instruments, protocols or origins')
        if s.get('model') in seen:
            raise ValueError('Do not choose or combine multiple attempts for one model')
        seen.add(s['model'])
    return summaries

# Capture this actual executed IPython cell for the pre-call freeze check.
import linecache as _freeze_linecache, sys as _freeze_sys
EXECUTED_SOURCE_CELLS['final_contract.py'] = ''.join(_freeze_linecache.getlines(_freeze_sys._getframe().f_code.co_filename))


In [ ]:
"""Bounded work, atomic evidence and interruption-aware coverage for final-v1.

No model/SDK is imported here. Caller supplies the authorized provider dispatcher.
Workers must implement the protocol's finite HTTP timeout; Python cannot forcibly
terminate an arbitrary running thread. On interruption we wait for active workers.
"""
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from contextvars import copy_context
from datetime import datetime, timezone
import json
from pathlib import Path
import time
from uuid import uuid4


class IncompleteRun(RuntimeError):
    """A partial/failed attempt has coverage evidence but no total score."""


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def atomic_json(path: Path, value: dict) -> None:
    path = Path(path)
    tmp = path.with_name(path.name + '.' + uuid4().hex + '.tmp')
    try:
        tmp.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False)+'\n', encoding='utf-8')
        tmp.replace(path)
    finally:
        if tmp.exists():
            tmp.unlink()


def _preserve_error_write(path, data, original):
    try:
        atomic_json(path, data)
    except BaseException as write_error:
        if original is None:
            raise
        # Preserve the provider/interruption exception, not a secondary disk error.
        note = f'Evidence write also failed: {type(write_error).__name__}'
        if hasattr(original, 'add_note'):
            original.add_note(note)


def run_one_case(case, manifest, folder, prompt_provider):
    folder = Path(folder)
    dst = folder/'records'/(case['id']+'.json')
    if dst.exists():
        raise FileExistsError('A case in this attempt already has evidence')
    base = record_base(case, manifest)
    record = dict(base, started_at=utc_now(), run_status='started')
    started = time.monotonic()
    error = None
    try:
        raw = prompt_provider(case['id'], base['prompt'])
        record.update(make_record(case, manifest, raw))
        return record
    except BaseException as exc:
        error = exc
        record.update(run_status='infrastructure_error' if isinstance(exc, Exception) else 'interrupted',
                      exception_type=type(exc).__name__)
        raise
    finally:
        record.update(finished_at=utc_now(), elapsed_seconds=time.monotonic()-started)
        _preserve_error_write(dst, record, error)


def dispatch_threads(items, worker, *, workers=2):
    """At most two active jobs; collect failures without retry; drain on exit.

    Ordinary failures are persisted by run_one_case and do not stop other items.
    BaseException interruptions stop further scheduling and cancel queued work.
    """
    if type(workers) is not int or workers not in (1, 2):
        raise ValueError('Only one or two workers are permitted')
    items = iter(items)
    pool = ThreadPoolExecutor(max_workers=workers, thread_name_prefix='fechamento-final')
    pending = set()
    outcomes = []
    try:
        for _ in range(workers):
            item = next(items, None)
            if item is not None:
                pending.add(pool.submit(copy_context().run, worker, item))
        while pending:
            done, pending = wait(pending, return_when=FIRST_COMPLETED)
            # Process completed jobs before scheduling their replacements.
            for future in done:
                try:
                    future.result()
                    outcomes.append('returned')
                except Exception as exc:
                    outcomes.append(type(exc).__name__)
            for _ in done:
                item = next(items, None)
                if item is not None:
                    pending.add(pool.submit(copy_context().run, worker, item))
        return outcomes
    finally:
        for future in pending:
            future.cancel()
        pool.shutdown(wait=True, cancel_futures=True)


def read_records(folder):
    records = []
    for p in sorted((Path(folder)/'records').glob('*.json')):
        r = json.loads(p.read_text(encoding='utf-8'))
        if r.get('case_id') != p.stem:
            raise ValueError('Record filename differs from case ID')
        records.append(r)
    return records


def run_attempt(cases, manifest, output_root, dispatcher):
    validate_manifest(manifest, cases)
    # Deep copy prevents caller mutation from changing a running attempt.
    cases = json.loads(json.dumps(cases, ensure_ascii=False))
    manifest = json.loads(json.dumps(manifest, ensure_ascii=False))
    folder = Path(output_root)/manifest['run_id']
    folder.mkdir(parents=True, exist_ok=False)
    (folder/'records').mkdir()
    atomic_json(folder/'manifest.json', manifest)
    state = {key: manifest[key] for key in ('run_id','model','evidence_kind','dataset_sha256','instrument_sha256','protocol_sha256','manifest_sha256')}
    state.update(state='running', started_at=utc_now(), planned_cases=len(cases), completed_cases=0,
                 failed_cases=0, missing_cases=len(cases), dispatch_finished=False)
    atomic_json(folder/'status.json', state)
    error = None
    try:
        dispatcher(cases, manifest, folder)
        state['dispatch_finished'] = True
        records = read_records(folder)
        try:
            summary = summarize_final(records, cases, manifest)
        except ValueError as exc:
            raise IncompleteRun('Incomplete or invalid evidence; no total score') from exc
        atomic_json(folder/'summary.json', summary)
        state['state'] = 'completed'
        return summary
    except BaseException as exc:
        error = exc
        state.update(state='incomplete_no_score' if isinstance(exc, Exception) else 'interrupted_no_score',
                     exception_type=type(exc).__name__)
        raise
    finally:
        try:
            records = read_records(folder)
            state['completed_cases'] = sum(r.get('run_status') == 'completed' for r in records)
            state['failed_cases'] = sum(r.get('run_status') != 'completed' for r in records)
            state['missing_cases'] = len(cases)-len(records)
        except BaseException as reconciliation_error:
            state['reconciliation_error'] = type(reconciliation_error).__name__
            if error is None:
                state['state'] = 'invalid_evidence_no_score'
                _preserve_error_write(folder/'status.json', state, reconciliation_error)
                raise
        state['finished_at'] = utc_now()
        _preserve_error_write(folder/'status.json', state, error)

# Capture this actual executed IPython cell for the pre-call freeze check.
import linecache as _freeze_linecache, sys as _freeze_sys
EXECUTED_SOURCE_CELLS['final_runner.py'] = ''.join(_freeze_linecache.getlines(_freeze_sys._getframe().f_code.co_filename))


In [ ]:
"""Register final-v1 tasks with the native Kaggle SDK; offline by default at import.

Real execution uses only Kaggle's authenticated model proxy, never personal API
keys. Tests use an explicit offline_fixture label and a simulated SDK.
"""
import copy
from datetime import datetime, timezone
import os
from pathlib import Path
from uuid import uuid4


def _concrete_return(tp):
    def annotate(fn):
        fn.__annotations__['return'] = tp
        return fn
    return annotate


def require_uncached_sdk(kbench):
    """Fail closed unless the native SDK explicitly reports cache disabled.

    KaggleClient.use_cache is a public flag in the SDK's official source. This
    checks that flag without altering global configuration. Provider-side prompt
    caching is outside this client control and is not claimed to be disabled.
    """
    if getattr(getattr(kbench, 'client', None), 'use_cache', None) is not False:
        raise RuntimeError('SDK cache must be explicitly false; unknown/enabled cache is not allowed')


def register_final_task(kbench, cases, source_hashes,
                        output_root='/kaggle/working/fechamento_final_v1', *,
                        evidence_kind='kaggle_model_run'):
    validate_final_cases(cases)
    require_uncached_sdk(kbench)
    if evidence_kind not in {'kaggle_model_run', 'offline_fixture'}:
        raise ValueError('Unsupported evidence origin')
    # Freeze the definitions read by the registration; the manifest will bind them.
    cases = copy.deepcopy(cases)
    source_hashes = copy.deepcopy(source_hashes)

    @kbench.task(name='fechamento_br_final_row_v1', store_task=False, store_run=False)
    @_concrete_return(dict)
    def final_row(llm, case, manifest, folder) -> dict:
        if llm.name != manifest['model']:
            raise RuntimeError('Actual model name does not match attempt manifest')
        settings = manifest['model_settings']
        def prompt_provider(case_id, prompt):
            return llm.prompt(prompt, seed=0, temperature=settings['temperature_passed'])
        # Task.run creates the SDK's isolated orphan conversation for this row.
        return run_one_case(case, manifest, folder, prompt_provider)

    @kbench.task(name='fechamento_br_final_v1', version=1,
                 description='Context-sensitive normalization: fixed contrast pairs and controls with deterministic grading.')
    @_concrete_return(float)
    def final_task(llm) -> float:
        protocol = default_protocol()
        require_uncached_sdk(kbench)
        if evidence_kind == 'kaggle_model_run':
            if not Path('/kaggle/working').is_dir() or os.environ.get('KAGGLE_USE_MODEL_PROXY', '').lower() != 'true':
                raise RuntimeError('Native free Kaggle proxy required; no personal API fallback')
            if getattr(llm, 'name', None) not in protocol['model_order']:
                raise RuntimeError('Model is outside the predeclared final selection')
        client = getattr(llm, 'client', None)
        if client is None or not callable(getattr(client, 'with_options', None)):
            raise RuntimeError('Cannot enforce the declared HTTP timeout/retry policy')
        model = copy.copy(llm)
        model.client = client.with_options(timeout=protocol['http_timeout_seconds'], max_retries=0)
        model.stream_responses = False
        support = getattr(model, 'support_temperature', None)
        # None is explicitly reported as unknown; no claim of deterministic sampling.
        settings = {'support_temperature': support,
                    'temperature_passed': 0 if support is True else None,
                    'seed_requested': 0, 'effective_seed': 'not_verified_by_provider',
                    'stream': False, 'http_timeout_seconds':120, 'client_retries':0,
                    'sdk_cache_enabled_observed': False,
                    'provider_prompt_cache': 'not_controlled_by_this_instrument'}
        run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')+'_'+uuid4().hex[:12]
        manifest = make_manifest(cases, model=model.name, run_id=run_id,
                                 evidence_kind=evidence_kind, source_hashes=source_hashes,
                                 protocol=protocol, model_settings=settings)
        def dispatch(items, m, folder):
            def worker(case):
                # Native child task preserves the actual chat trace in the SDK.
                require_uncached_sdk(kbench)
                return final_row.run(llm=model, case=case, manifest=m, folder=str(folder))
            return dispatch_threads(items, worker, workers=protocol['n_jobs'])
        summary = run_attempt(cases, manifest, output_root, dispatch)
        print('FINAL_ATTEMPT_COMPLETED', model.name, run_id,
              summary['primary_correct'], '/', summary['planned_cases'], flush=True)
        return float(summary['score'])

    return final_task

# Capture this actual executed IPython cell for the pre-call freeze check.
import linecache as _freeze_linecache, sys as _freeze_sys
EXECUTED_SOURCE_CELLS['kaggle_final_adapter.py'] = ''.join(_freeze_linecache.getlines(_freeze_sys._getframe().f_code.co_filename))


In [ ]:
CANDIDATE_TEXT = '{\n  "version": "final-v1-candidate",\n  "purpose": "new_final_corpus_pending_independent_review",\n  "authored_at_utc": "2026-10-07T18:58:20.061791+00:00",\n  "author": "ChatGPT executor; human owner Gabriel",\n  "review_status": "pending Codex Desktop 6.1 Sol Max; no final model calls permitted before review",\n  "design": "28 cases = 12 contrast pairs + 4 controls; seven cases per family; fixed interleaved order",\n  "cases": [\n    {\n      "id": "N01A",\n      "role": "contrast",\n      "pair": "N01",\n      "group": "numeros",\n      "input": {\n        "kind": "money",\n        "raw": "4.700",\n        "context": "Valor monetário não negativo. A fonte declara pt-BR: ponto separa milhares e vírgula separa decimais. Não arredondar; zeros decimais excedentes podem ser removidos."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "4700.00"\n      },\n      "rationale": "O ponto é separador de milhar porque a fonte declara pt-BR."\n    },\n    {\n      "id": "A01A",\n      "role": "contrast",\n      "pair": "A01",\n      "group": "ausencias",\n      "input": {\n        "kind": "count",\n        "raw": "",\n        "context": "Quantidade inteira não negativa. A gramática aceita apenas dígitos ASCII de 0 a 9. Campo vazio significa não informado; zero explícito é válido. Zeros iniciais são permitidos na entrada."\n      },\n      "expected": {\n        "status": "missing",\n        "value": null\n      },\n      "rationale": "Nenhuma quantidade foi informada."\n    },\n    {\n      "id": "D01A",\n      "role": "contrast",\n      "pair": "D01",\n      "group": "datas",\n      "input": {\n        "kind": "date",\n        "raw": "29/02/2400",\n        "context": "Data no calendário gregoriano. A origem declara DD/MM/AAAA. Não substituir nem corrigir componentes."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "2400-02-29"\n      },\n      "rationale": "O ano 2400 é divisível por 400 e é bissexto no calendário gregoriano."\n    },\n    {\n      "id": "I01A",\n      "role": "contrast",\n      "pair": "I01",\n      "group": "identificadores",\n      "input": {\n        "kind": "identifier",\n        "raw": "00072",\n        "context": "Identificador textual opaco, não uma quantidade. Remova somente espaços e tabulações externos. Preserve todos os caracteres internos, inclusive zeros e espaços. Não interprete como número nem como notação científica."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "00072"\n      },\n      "rationale": "Zeros fazem parte do identificador, por isso todos são mantidos."\n    },\n    {\n      "id": "N01B",\n      "role": "contrast",\n      "pair": "N01",\n      "group": "numeros",\n      "input": {\n        "kind": "money",\n        "raw": "4.700",\n        "context": "Valor monetário não negativo. A fonte declara en-US: vírgula separa milhares e ponto separa decimais. Não arredondar; zeros decimais excedentes podem ser removidos."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "4.70"\n      },\n      "rationale": "O ponto é decimal porque a fonte declara en-US; o zero decimal final pode ser removido."\n    },\n    {\n      "id": "A01B",\n      "role": "contrast",\n      "pair": "A01",\n      "group": "ausencias",\n      "input": {\n        "kind": "count",\n        "raw": "000",\n        "context": "Quantidade inteira não negativa. A gramática aceita apenas dígitos ASCII de 0 a 9. Campo vazio significa não informado; zero explícito é válido. Zeros iniciais são permitidos na entrada."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "0"\n      },\n      "rationale": "A sequência de zeros informa explicitamente a quantidade zero."\n    },\n    {\n      "id": "D01B",\n      "role": "contrast",\n      "pair": "D01",\n      "group": "datas",\n      "input": {\n        "kind": "date",\n        "raw": "29/02/2300",\n        "context": "Data no calendário gregoriano. A origem declara DD/MM/AAAA. Não substituir nem corrigir componentes."\n      },\n      "expected": {\n        "status": "invalid",\n        "value": null\n      },\n      "rationale": "2300 é divisível por 100, mas não por 400; não existe 29 de fevereiro nesse ano."\n    },\n    {\n      "id": "I01B",\n      "role": "contrast",\n      "pair": "I01",\n      "group": "identificadores",\n      "input": {\n        "kind": "count",\n        "raw": "00072",\n        "context": "Quantidade inteira não negativa. A gramática aceita apenas dígitos ASCII de 0 a 9. Campo vazio significa não informado; zero explícito é válido. Zeros iniciais são permitidos na entrada."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "72"\n      },\n      "rationale": "A mesma cadeia informa uma quantidade decimal, não um código."\n    },\n    {\n      "id": "N02A",\n      "role": "contrast",\n      "pair": "N02",\n      "group": "numeros",\n      "input": {\n        "kind": "money",\n        "raw": "6,250",\n        "context": "Valor monetário não negativo. A fonte declara pt-BR: ponto separa milhares e vírgula separa decimais. Não arredondar; zeros decimais excedentes podem ser removidos."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "6.25"\n      },\n      "rationale": "A vírgula decimal declarada produz 6.250; retirar apenas o zero final mantém o valor."\n    },\n    {\n      "id": "A02A",\n      "role": "contrast",\n      "pair": "A02",\n      "group": "ausencias",\n      "input": {\n        "kind": "count",\n        "raw": "ND",\n        "context": "Quantidade inteira não negativa. A gramática aceita apenas dígitos ASCII de 0 a 9. Campo vazio significa não informado; zero explícito é válido. Zeros iniciais são permitidos na entrada. O marcador literal ND foi definido pela fonte como não informado."\n      },\n      "expected": {\n        "status": "missing",\n        "value": null\n      },\n      "rationale": "A fonte autoriza expressamente ND como marcador de ausência."\n    },\n    {\n      "id": "D02A",\n      "role": "contrast",\n      "pair": "D02",\n      "group": "datas",\n      "input": {\n        "kind": "date",\n        "raw": "06/09/2032",\n        "context": "Data no calendário gregoriano. A origem declara DD/MM/AAAA. Não substituir nem corrigir componentes."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "2032-09-06"\n      },\n      "rationale": "DD/MM determina seis de setembro, sem ambiguidade residual."\n    },\n    {\n      "id": "I02A",\n      "role": "contrast",\n      "pair": "I02",\n      "group": "identificadores",\n      "input": {\n        "kind": "identifier",\n        "raw": "  RQ09  ",\n        "context": "Identificador textual opaco, não uma quantidade. Remova somente espaços e tabulações externos. Preserve todos os caracteres internos, inclusive zeros e espaços. Não interprete como número nem como notação científica."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "RQ09"\n      },\n      "rationale": "Somente os espaços externos são removidos."\n    },\n    {\n      "id": "N02B",\n      "role": "contrast",\n      "pair": "N02",\n      "group": "numeros",\n      "input": {\n        "kind": "money",\n        "raw": "6,250",\n        "context": "Valor monetário não negativo. A origem pode ser pt-BR ou en-US e não há metadados para escolher. Não deduza a origem pelo idioma. Não arredondar; zeros decimais excedentes podem ser removidos."\n      },\n      "expected": {\n        "status": "ambiguous",\n        "value": null\n      },\n      "rationale": "Duas leituras válidas permanecem: 6.25 em pt-BR e 6250.00 em en-US."\n    },\n    {\n      "id": "A02B",\n      "role": "contrast",\n      "pair": "A02",\n      "group": "ausencias",\n      "input": {\n        "kind": "count",\n        "raw": "ND",\n        "context": "Quantidade inteira não negativa. A gramática aceita apenas dígitos ASCII de 0 a 9. Campo vazio significa não informado; zero explícito é válido. Zeros iniciais são permitidos na entrada. A fonte não definiu marcadores de ausência; ND não tem significado especial."\n      },\n      "expected": {\n        "status": "invalid",\n        "value": null\n      },\n      "rationale": "Sem definição de marcador, ND viola a gramática de quantidade; não adivinhar sua intenção."\n    },\n    {\n      "id": "D02B",\n      "role": "contrast",\n      "pair": "D02",\n      "group": "datas",\n      "input": {\n        "kind": "date",\n        "raw": "06/09/2032",\n        "context": "Data no calendário gregoriano. A origem declara MM/DD/AAAA. Não substituir nem corrigir componentes."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "2032-06-09"\n      },\n      "rationale": "MM/DD determina nove de junho; a mesma cadeia tem outro valor pela origem explícita."\n    },\n    {\n      "id": "I02B",\n      "role": "contrast",\n      "pair": "I02",\n      "group": "identificadores",\n      "input": {\n        "kind": "identifier",\n        "raw": "  RQ  09  ",\n        "context": "Identificador textual opaco, não uma quantidade. Remova somente espaços e tabulações externos. Preserve todos os caracteres internos, inclusive zeros e espaços. Não interprete como número nem como notação científica."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "RQ  09"\n      },\n      "rationale": "Dois espaços internos são parte do código e devem permanecer."\n    },\n    {\n      "id": "N03A",\n      "role": "contrast",\n      "pair": "N03",\n      "group": "numeros",\n      "input": {\n        "kind": "money",\n        "raw": "28,4300",\n        "context": "Valor monetário não negativo. A fonte declara pt-BR: ponto separa milhares e vírgula separa decimais. Não arredondar; zeros decimais excedentes podem ser removidos."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "28.43"\n      },\n      "rationale": "Só há zeros além da segunda casa; a redução é exata."\n    },\n    {\n      "id": "A03A",\n      "role": "contrast",\n      "pair": "A03",\n      "group": "ausencias",\n      "input": {\n        "kind": "count",\n        "raw": "\\t  ",\n        "context": "Quantidade inteira não negativa. A gramática aceita apenas dígitos ASCII de 0 a 9. Campo vazio significa não informado; zero explícito é válido. Zeros iniciais são permitidos na entrada. Remova espaços e tabulações externos antes de validar; se o conteúdo restante for vazio, está não informado."\n      },\n      "expected": {\n        "status": "missing",\n        "value": null\n      },\n      "rationale": "Depois da remoção externa expressamente permitida, não resta conteúdo."\n    },\n    {\n      "id": "D03A",\n      "role": "contrast",\n      "pair": "D03",\n      "group": "datas",\n      "input": {\n        "kind": "date",\n        "raw": "07/11/2031",\n        "context": "Data no calendário gregoriano. A origem declara DD/MM/AAAA. Não substituir nem corrigir componentes."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "2031-11-07"\n      },\n      "rationale": "O formato DD/MM explícito determina sete de novembro."\n    },\n    {\n      "id": "I03A",\n      "role": "contrast",\n      "pair": "I03",\n      "group": "identificadores",\n      "input": {\n        "kind": "identifier",\n        "raw": "5E03",\n        "context": "Identificador textual opaco, não uma quantidade. Remova somente espaços e tabulações externos. Preserve todos os caracteres internos, inclusive zeros e espaços. Não interprete como número nem como notação científica."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "5E03"\n      },\n      "rationale": "Código alfanumérico é preservado literalmente; não transformá-lo em 5000."\n    },\n    {\n      "id": "N03B",\n      "role": "contrast",\n      "pair": "N03",\n      "group": "numeros",\n      "input": {\n        "kind": "money",\n        "raw": "28,4307",\n        "context": "Valor monetário não negativo. A fonte declara pt-BR: ponto separa milhares e vírgula separa decimais. Não arredondar; zeros decimais excedentes podem ser removidos."\n      },\n      "expected": {\n        "status": "invalid",\n        "value": null\n      },\n      "rationale": "O dígito 7 não nulo além das duas casas impede representação exata; truncar ou arredondar é proibido."\n    },\n    {\n      "id": "A03B",\n      "role": "contrast",\n      "pair": "A03",\n      "group": "ausencias",\n      "input": {\n        "kind": "count",\n        "raw": "\\t 07 ",\n        "context": "Quantidade inteira não negativa. A gramática aceita apenas dígitos ASCII de 0 a 9. Campo vazio significa não informado; zero explícito é válido. Zeros iniciais são permitidos na entrada. Remova espaços e tabulações externos antes de validar; se o conteúdo restante for vazio, está não informado."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "7"\n      },\n      "rationale": "A remoção externa deixa 07, quantidade decimal inteira igual a sete."\n    },\n    {\n      "id": "D03B",\n      "role": "contrast",\n      "pair": "D03",\n      "group": "datas",\n      "input": {\n        "kind": "date",\n        "raw": "07/11/2031",\n        "context": "Data no calendário gregoriano. A origem não informa se usa DD/MM/AAAA ou MM/DD/AAAA. Ambas são possíveis; não escolha pelo idioma desta pergunta. Não substituir componentes."\n      },\n      "expected": {\n        "status": "ambiguous",\n        "value": null\n      },\n      "rationale": "Sete de novembro e onze de julho são datas válidas; falta metadado para escolher."\n    },\n    {\n      "id": "I03B",\n      "role": "contrast",\n      "pair": "I03",\n      "group": "identificadores",\n      "input": {\n        "kind": "count",\n        "raw": "5E03",\n        "context": "Quantidade inteira não negativa. A gramática aceita apenas dígitos ASCII de 0 a 9. Campo vazio significa não informado; zero explícito é válido. Zeros iniciais são permitidos na entrada."\n      },\n      "expected": {\n        "status": "invalid",\n        "value": null\n      },\n      "rationale": "A gramática de quantidade permite só dígitos; não converter silenciosamente notação científica."\n    },\n    {\n      "id": "NCTL",\n      "role": "control",\n      "pair": null,\n      "group": "numeros",\n      "input": {\n        "kind": "money",\n        "raw": "84,65",\n        "context": "Valor monetário não negativo. A fonte declara pt-BR: ponto separa milhares e vírgula separa decimais. Não arredondar; zeros decimais excedentes podem ser removidos."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "84.65"\n      },\n      "rationale": "Controle ordinário: decimal pt-BR com duas casas já fornecidas."\n    },\n    {\n      "id": "ACTL",\n      "role": "control",\n      "pair": null,\n      "group": "ausencias",\n      "input": {\n        "kind": "count",\n        "raw": "18",\n        "context": "Quantidade inteira não negativa. A gramática aceita apenas dígitos ASCII de 0 a 9. Campo vazio significa não informado; zero explícito é válido. Zeros iniciais são permitidos na entrada."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "18"\n      },\n      "rationale": "Controle positivo simples: dezoito foi informado explicitamente."\n    },\n    {\n      "id": "DCTL",\n      "role": "control",\n      "pair": null,\n      "group": "datas",\n      "input": {\n        "kind": "date",\n        "raw": "16/08/2029",\n        "context": "Data no calendário gregoriano. A origem declara DD/MM/AAAA. Não substituir nem corrigir componentes."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "2029-08-16"\n      },\n      "rationale": "Controle sem fronteira: dezesseis de agosto é uma data válida no formato declarado."\n    },\n    {\n      "id": "ICTL",\n      "role": "control",\n      "pair": null,\n      "group": "identificadores",\n      "input": {\n        "kind": "identifier",\n        "raw": "9007199254741017",\n        "context": "Identificador textual opaco, não uma quantidade. Remova somente espaços e tabulações externos. Preserve todos os caracteres internos, inclusive zeros e espaços. Não interprete como número nem como notação científica."\n      },\n      "expected": {\n        "status": "ok",\n        "value": "9007199254741017"\n      },\n      "rationale": "Código longo permanece como texto integral, sem conversão de ponto flutuante."\n    }\n  ]\n}\n'
FINAL_DOCUMENT = json.loads(CANDIDATE_TEXT)
FINAL_CASES = FINAL_DOCUMENT["cases"]
validate_final_cases(FINAL_CASES)
assert len(FINAL_CASES) == 28
assert sum(c["role"] == "control" for c in FINAL_CASES) == 4
print("OFFLINE_DEFINITIONS_READY: 28 synthetic cases; no model calls.")


In [ ]:
def verify_final_freeze():
    """Recompute the loaded corpus, source payloads, instructions and protocol.

    This catches accidental drift, not adversarial rewriting of the verifier or
    arbitrary Python monkey-patching. The separate review receipt records the
    exact builder/notebook bytes in addition to this runtime bundle.
    """
    if not FINAL_REVIEW_APPROVED:
        raise RuntimeError('Independent review not approved')
    if set(EXECUTED_SOURCE_CELLS) != set(SOURCE_HASHES) or not all(EXECUTED_SOURCE_CELLS.values()):
        raise RuntimeError('Cannot verify actual executed source cells; use the checked IPython runtime')
    actual_cell_hashes={name:hashlib.sha256(text.encode('utf-8')).hexdigest()
                        for name,text in EXECUTED_SOURCE_CELLS.items()}
    if actual_cell_hashes != REVIEWED_RECIPE['codecell_hashes'] or SOURCE_HASHES != REVIEWED_RECIPE['sources']:
        raise RuntimeError('Actual executed source differs from reviewed source cells')
    if json.loads(CANDIDATE_TEXT) != FINAL_DOCUMENT or FINAL_CASES != FINAL_DOCUMENT.get('cases'):
        raise RuntimeError('Runtime cases/document changed after the reviewed payload was loaded')
    actual_recipe={'sources':SOURCE_HASHES,'codecell_hashes':actual_cell_hashes,
                   'candidate_sha256':hashlib.sha256(CANDIDATE_TEXT.encode('utf-8')).hexdigest(),
                   'document_sha256':digest(FINAL_DOCUMENT),
                   'protocol_sha256':digest(default_protocol()),
                   'prompt_template_sha256':sha256_text(FINAL_INSTRUCTIONS),
                   'builder_sha256':REVIEWED_RECIPE['builder_sha256']}
    actual_bundle=hashlib.sha256(json.dumps(actual_recipe,sort_keys=True).encode()).hexdigest()
    if actual_recipe != REVIEWED_RECIPE or actual_bundle != BUNDLE_SHA256 or actual_bundle != APPROVED_BUNDLE_SHA256:
        raise RuntimeError('Recomputed runtime bundle does not match the approved review receipt')
    validate_final_cases(FINAL_CASES)


In [ ]:
# Do not change the dataset, grader, prompts or protocol after measuring.
if not RUN_FINAL_MODELS:
    print('FINAL_MODEL_CALLS_DISABLED')
else:
    verify_final_freeze()
    if not Path('/kaggle/working').is_dir() or os.environ.get('KAGGLE_USE_MODEL_PROXY','').lower() != 'true':
        raise RuntimeError('Native Kaggle free proxy required; no paid fallback')
    import kaggle_benchmarks as kbench
    from importlib.metadata import version
    sdk_version = version('kaggle-benchmarks')
    print('ACTUAL_KAGGLE_SDK_VERSION', sdk_version)
    if sdk_version != '0.6.1':
        raise RuntimeError('SDK differs from protocol validation: review compatibility before calls')
    require_uncached_sdk(kbench)
    output_root=Path('/kaggle/working/fechamento_final_v1')
    fechamento_br_final_v1=register_final_task(kbench,FINAL_CASES,SOURCE_HASHES,output_root)
    model_ids=default_protocol()['model_order']
    if any(mid not in kbench.llms for mid in model_ids):
        raise RuntimeError('A predeclared model is unavailable; do not replace based on scores')
    for mid in model_ids:
        previous=[]
        if output_root.exists():
            previous=[json.loads(p.read_text()) for p in output_root.glob('*/manifest.json')]
        if any(m.get('model')==mid and m.get('dataset_sha256')==digest(FINAL_CASES) for m in previous):
            print('SKIP_EXISTING_FINAL_ATTEMPT',mid)
            continue
        print('BEGIN_FINAL_MODEL',mid,flush=True)
        try:
            require_uncached_sdk(kbench)
            native_run=fechamento_br_final_v1.run(llm=kbench.llms[mid])
            print('NATIVE_RUN_RETURNED',mid,str(native_run.status),flush=True)
        except Exception as exc:
            print('FINAL_ATTEMPT_EXCEPTION',mid,type(exc).__name__,flush=True)
        print('END_FINAL_MODEL',mid,flush=True)
    print('ALL_AUTHORIZED_FINAL_ATTEMPTS_FINISHED',flush=True)


In [ ]:
# Archive only our native result files and our attempt folders. No deletion or publication.
if RUN_FINAL_MODELS and Path('/kaggle/working/fechamento_final_v1').is_dir():
    import shutil,zipfile
    wd=Path('/kaggle/working')
    destination=wd/'fechamento_final_native_archive';destination.mkdir(exist_ok=True)
    for p in wd.glob('fechamento_br_final_v1*.json'):
        if p.name.endswith(('.run.json','.task.json','.result.json','.atif.json')):
            target=destination/p.name
            if target.exists() and target.read_bytes()!=p.read_bytes():
                raise RuntimeError('Refuse to overwrite previous native evidence')
            if not target.exists():shutil.copyfile(p,target)
    zip_path=wd/'Fechamento_BR_Final_Evidence.zip'
    with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as archive:
        for folder in [wd/'fechamento_final_v1',destination]:
            for p in sorted(folder.rglob('*.json')):
                archive.write(p,p.relative_to(wd).as_posix())
    print('EVIDENCE_ARCHIVE',str(zip_path))
    print('No Build Task, choose, article or public submission action was performed.')


## Interpretation and provenance

The 28 cases are constructed, not a random sample of real administrative records.
Pairs share inputs or rules and are correlated. Present descriptive counts, not
confidence intervals assuming 28 independent draws. Sampling was requested as
seed=0 and temperature=0 when supported; unsupported temperature is recorded
as None and effective seed support is unknown. No claim of determinism is made.

Each attempt records an immutable manifest, original responses and coverage.
The SHA-256 chain detects accidental mixing/tampering; it does not externally
authenticate a provider or the time when a human authored gold labels.

Final task publication and the required English DEV article are separate steps.
A previously recorded tool safety block on publication must not be bypassed.
Sources: official Kaggle Benchmarks quick_start.md and cookbook.md in the Kaggle
GitHub repository, and dev.to/challenges/kaggle-2026-09-23.
